# SC4001 Topic J: Member B training (CLIP probe + ResNet-50)

**Runtime:** set Runtime > Change runtime type > **T4 GPU** before starting.

Run the cells top to bottom. Every output (checkpoints, predictions, experiment log) is saved to
`MyDrive/sc4001_project/results/`, so nothing is lost if Colab disconnects. If it disconnects, rerun
cells 1–3 and then the training cell: runs that already finished are skipped automatically.

### 1. Mount Drive, get the latest code, install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone https://github.com/ster2345/SC4001-Grp-Proj.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

Mounted at /content/drive
Cloning into '/content/repo'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 73 (delta 30), reused 39 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 212.98 KiB | 8.52 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/repo


### 2. Unpack both data versions from Drive to Colab's local disk

In [ ]:
if not os.path.isdir('/content/data/original'):
    !mkdir -p /content/data
    !tar -xf /content/drive/MyDrive/sc4001_project/raw_downloads/original_subset.tar -C /content/data
if not os.path.isdir('/content/data/cleaned'):
    !tar -xf /content/drive/MyDrive/sc4001_project/raw_downloads/cleaned_subset.tar -C /content/data
!python src/check_splits.py --csv splits/splits.csv --root /content/data   # expect: ALL CHECKS PASSED: 5500 images

ALL CHECKS PASSED: 5500 images, {'test': np.int64(3000), 'train': np.int64(2000), 'val': np.int64(500)}


### 3. Check the GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch runtime to T4 GPU')

GPU: Tesla T4


### 4. Quick code check (about 1 minute)
Trains on 64 images for 1 epoch, saved to a throwaway folder (not Drive). It only checks that the code runs:
the numbers mean nothing.

In [ ]:
!python src/train_resnet.py --version original --seed 42 --quick --out /content/quick_test
!python src/train_resnet.py --version cleaned  --seed 42 --quick --out /content/quick_test

I0000 00:00:1794151903.348345    1403 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1794151904.148402    1403 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1794151904.151469    1403 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
I0000 00:00:1794151907.073831    1403 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.116. The framework PJRT API version is 0.116.
I0000 00:00:1794151907.675399    1403 pjrt_c_api_client.cc:187] PjRtCApiClient created.
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 258MB/s]
epoch  1  train_loss 0.6885  val_loss 0.7105  val_acc 0.4375  val_auc

### 5. CLIP linear probe (both data versions, about 1 minute)
Logistic regression on the saved CLIP features. It prints validation results for each C and picks the best.

In [ ]:
!python src/train_clip_probe.py --version original
!python src/train_clip_probe.py --version cleaned

### 6. ResNet-50: 2 data versions × 3 seeds = 6 runs
Each run prints one line per epoch. Runs whose predictions already exist on Drive are skipped,
so after a disconnect you can just rerun this cell.

In [ ]:
RES = '/content/drive/MyDrive/sc4001_project/results/predictions'
for version in ['original', 'cleaned']:
    for seed in [42, 43, 44]:
        done = os.path.exists(f'{RES}/resnet50_{version}_seed{seed}.csv')
        if done:
            print(f'skip {version} seed {seed}: already done')
            continue
        print(f'===== ResNet-50 | {version} | seed {seed} =====')
        !python src/train_resnet.py --version {version} --seed {seed}

### 7. Experiment log (validation results only) and prediction-file check

In [ ]:
import pandas as pd, glob, sys
sys.path.insert(0, 'src')
from train_utils import check_prediction_file
print(pd.read_csv('/content/drive/MyDrive/sc4001_project/results/experiment_log.csv').to_string())
for f in sorted(glob.glob(f'{RES}/*.csv')):
    check_prediction_file(f)

### 8. Ablation: more epochs (15 instead of 10), 2 versions × 3 seeds (about 30 min)
Checks whether the ResNet was undertrained (some main runs picked epoch 8 of 10). Everything else is unchanged.
Saved as `resnet50_<version>_seed<s>_ep15.*`, so the main results are **not** overwritten.
Already-finished ablation runs are skipped, so you can rerun this cell after a disconnect.

In [ ]:
   RES = '/content/drive/MyDrive/sc4001_project/results/predictions'
   import pandas as pd

In [ ]:
for version in ['original', 'cleaned']:
    for seed in [42, 43, 44]:
        if os.path.exists(f'{RES}/resnet50_{version}_seed{seed}_ep15.csv'):
            print(f'skip {version} seed {seed} ep15: already done')
            continue
        print(f'===== ABLATION ep15 | ResNet-50 | {version} | seed {seed} =====')
        !python src/train_resnet.py --version {version} --seed {seed} --max-epochs 15 --tag ep15

===== ABLATION ep15 | ResNet-50 | original | seed 42 =====
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 169MB/s]
epoch  1  train_loss 0.5366  val_loss 0.2783  val_acc 0.8820  val_auc 0.9619
epoch  2  train_loss 0.1815  val_loss 0.1701  val_acc 0.9480  val_auc 0.9896
epoch  3  train_loss 0.0936  val_loss 0.1314  val_acc 0.9540  val_auc 0.9924
epoch  4  train_loss 0.0410  val_loss 0.1150  val_acc 0.9560  val_auc 0.9938
epoch  5  train_loss 0.0345  val_loss 0.1172  val_acc 0.9600  val_auc 0.9917
epoch  6  train_loss 0.0273  val_loss 0.1206  val_acc 0.9600  val_auc 0.9936
epoch  7  train_loss 0.0228  val_loss 0.1132  val_acc 0.9560  val_auc 0.9926
epoch  8  train_loss 0.0269  val_loss 0.1054  val_acc 0.9660  val_auc 0.9929
epoch  9  train_loss 0.0339  val_loss 0.1145  val_acc 0.9620  val_auc 0.9925
epoch 10  train_loss 0.0128  val_loss 0.1076  val_acc 0.9500  val_auc 0.994

In [ ]:
# compare main (max 10 epochs) vs ablation (max 15 epochs) on VALIDATION results only
log = pd.read_csv('/content/drive/MyDrive/sc4001_project/results/experiment_log.csv')
r = log[log.model.isin(['resnet50', 'resnet50_ep15'])]
print(r[['model', 'data_version', 'seed', 'best_epoch', 'val_acc', 'val_auc', 'val_loss']].to_string(index=False))

        model data_version  seed  best_epoch  val_acc  val_auc  val_loss
     resnet50     original  42.0         8.0    0.966   0.9929    0.1054
     resnet50     original  43.0         8.0    0.948   0.9948    0.0972
     resnet50     original  44.0         5.0    0.966   0.9941    0.0967
     resnet50      cleaned  42.0         8.0    0.924   0.9825    0.1792
     resnet50      cleaned  43.0         6.0    0.940   0.9866    0.1524
     resnet50      cleaned  44.0         7.0    0.930   0.9848    0.1645
resnet50_ep15     original  42.0         8.0    0.966   0.9929    0.1054
resnet50_ep15     original  43.0        11.0    0.956   0.9963    0.0852
resnet50_ep15     original  44.0         5.0    0.966   0.9941    0.0967
resnet50_ep15      cleaned  42.0        11.0    0.938   0.9866    0.1608
resnet50_ep15      cleaned  43.0         6.0    0.940   0.9866    0.1524
resnet50_ep15      cleaned  44.0         7.0    0.930   0.9848    0.1645
